In [2]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="Khat865/ANUBIS-Skeleton",
    repo_type="dataset",
    local_dir="./anubis_dataset",
    allow_patterns=["anubis/*.npy", "anubis/*.pkl"],
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/home/alan/mydev/college/football-injury-ai/notebooks/anubis_dataset'

In [ ]:
from pathlib import Path
import numpy as np
import pickle

root = Path("./anubis_dataset/anubis")

# Check files exist and get sizes
for f in root.iterdir():
    print(f.name, f.stat().st_size / 1e9, "GB")

# Load shapes (memory-safe)
x_trn = np.load(root / "trn_data_all_action_front.npy", mmap_mode="r")
x_val = np.load(root / "val_data_all_action_back.npy", mmap_mode="r")

print("train shape:", x_trn.shape, "dtype:", x_trn.dtype)
print("val shape:  ", x_val.shape, "dtype:", x_val.dtype)

# Load labels
with open(root / "trn_label_all_action_front.pkl", "rb") as f:
    try:
        trn_names, trn_labels = pickle.load(f)
    except Exception:
        trn_names, trn_labels = pickle.load(f, encoding="latin1")

with open(root / "val_label_all_action_back.pkl", "rb") as f:
    try:
        val_names, val_labels = pickle.load(f)
    except Exception:
        val_names, val_labels = pickle.load(f, encoding="latin1")

print("\n--- Labels ---")
print("train samples:", len(trn_names), len(trn_labels))
print("train labels:  min", min(trn_labels), "max", max(trn_labels), "unique", len(set(trn_labels)))
print("val samples:  ", len(val_names), len(val_labels))
print("val labels:   min", min(val_labels), "max", max(val_labels), "unique", len(set(val_labels)))

# First few sample names
print("\nSample names (train):", trn_names[:5])

In [ ]:
import numpy as np
from collections import Counter

# Class counts
trn_counts = Counter(trn_labels)
val_counts = Counter(val_labels)

print("--- Train class distribution ---")
print("most common:", trn_counts.most_common(5))
print("least common:", trn_counts.most_common()[-5:])

print("\n--- Val class distribution ---")
print("most common:", val_counts.most_common(5))
print("least common:", val_counts.most_common()[-5:])

# Check for missing classes in val
all_classes = set(range(102))
trn_classes = set(trn_counts.keys())
val_classes = set(val_counts.keys())

missing_in_val = all_classes - val_classes
missing_in_trn = all_classes - trn_classes

print("\nClasses missing in val:", len(missing_in_val), sorted(missing_in_val) if missing_in_val else "None")
print("Classes missing in trn:", len(missing_in_trn), sorted(missing_in_trn) if missing_in_trn else "None")

In [ ]:
import numpy as np

def scan_split(x, batch_size=256, desc=""):
    """Scan for corrupted clips."""
    n_samples = x.shape[0]
    n_batches = (n_samples + batch_size - 1) // batch_size
    
    stats = {
        "all_zero": 0,
        "has_nan": 0,
        "has_inf": 0,
        "low_valid_frames": 0,
        "person0_empty": 0,
        "person1_empty": 0,
    }
    
    for i in range(n_batches):
        start = i * batch_size
        end = min(start + batch_size, n_samples)
        
        # Load one batch at a time
        batch = x[start:end]
        
        # Check each sample in batch
        for j in range(end - start):
            sample = batch[j]  # shape: (3, 300, 32, 2)
            
            # Person 0 and Person 1
            p0 = sample[:, :, :, 0]  # (3, 300, 32)
            p1 = sample[:, :, :, 1]  # (3, 300, 32)
            
            # All zero (person 0)
            if np.all(p0 == 0):
                stats["all_zero"] += 1
            
            # NaN / Inf
            if np.any(np.isnan(p0)) or np.any(np.isnan(p1)):
                stats["has_nan"] += 1
            if np.any(np.isinf(p0)) or np.any(np.isinf(p1)):
                stats["has_inf"] += 1
            
            # Valid frames (non-zero in at least one joint)
            valid_p0 = np.sum(np.any(p0 != 0, axis=(0, 2)))  # count frames with any non-zero
            if valid_p0 < 30:  # less than 10% of 300 frames
                stats["low_valid_frames"] += 1
            
            # Person occupancy
            if np.all(p0 == 0):
                stats["person0_empty"] += 1
            if np.all(p1 == 0):
                stats["person1_empty"] += 1
    
    # Print results
    print(f"\n{desc} Integrity Scan (n={n_samples})")
    for k, v in stats.items():
        pct = 100 * v / n_samples
        print(f"  {k}: {v} ({pct:.2f}%)")
    
    return stats

# Run on both splits
stats_trn = scan_split(x_trn, desc="Train")
stats_val = scan_split(x_val, desc="Val")

In [ ]:
import numpy as np

def count_valid_frames(x, batch_size=256):
    """Count valid (non-zero) frames per sample."""
    n_samples = x.shape[0]
    valid_counts = []
    
    for i in range(0, n_samples, batch_size):
        end = min(i + batch_size, n_samples)
        batch = x[i:end, :, :, :, 0]  # person 0 only
        
        # Valid = at least one joint has non-zero values
        valid = np.any(batch != 0, axis=(1, 2))  # (batch, frames)
        valid_counts.append(np.sum(valid, axis=1))
    
    return np.concatenate(valid_counts)

print("Counting valid frames (train)...")
valid_trn = count_valid_frames(x_trn)
print("Counting valid frames (val)...")
valid_val = count_valid_frames(x_val)

print("\n--- Valid frame counts ---")
print("Train: min", valid_trn.min(), "max", valid_trn.max(), "mean", valid_trn.mean(), "median", np.median(valid_trn))
print("Val:   min", valid_val.min(), "max", valid_val.max(), "mean", valid_val.mean(), "median", np.median(valid_val))

# Histogram
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(valid_trn, bins=50, alpha=0.7)
axes[0].set_title("Train: Valid frames per clip")
axes[0].axvline(60, color='r', linestyle='--', label='target=60')
axes[0].legend()

axes[1].hist(valid_val, bins=50, alpha=0.7)
axes[1].set_title("Val: Valid frames per clip")
axes[1].axvline(60, color='r', linestyle='--', label='target=60')
axes[1].legend()

plt.tight_layout()
plt.savefig("anubis_valid_frames_hist.png", dpi=100)
print("\nSaved: anubis_valid_frames_hist.png")

In [ ]:
import numpy as np
from collections import Counter

def analyze_person_occupancy(x, labels, names, batch_size=256):
    """Analyze person 1 usage."""
    n_samples = x.shape[0]
    
    both_present = 0
    only_p0 = 0
    only_p1 = 0
    both_empty = 0
    
    # Sample class-wise
    class_both = Counter()
    class_p0only = Counter()
    
    for i in range(0, n_samples, batch_size):
        end = min(i + batch_size, n_samples)
        batch = x[i:end]  # (batch, 3, 300, 32, 2)
        
        for j in range(end - i):
            p0 = batch[j, :, :, :, 0]
            p1 = batch[j, :, :, :, 1]
            
            p0_active = np.any(p0 != 0)
            p1_active = np.any(p1 != 0)
            
            label = labels[i + j]
            
            if p0_active and p1_active:
                both_present += 1
                class_both[label] += 1
            elif p0_active and not p1_active:
                only_p0 += 1
                class_p0only[label] += 1
            elif not p0_active and p1_active:
                only_p1 += 1
            else:
                both_empty += 1
    
    total = both_present + only_p0 + only_p1 + both_empty
    
    print(f"\nPerson Occupancy (n={total})")
    print(f"  Both present:  {both_present} ({100*both_present/total:.1f}%)")
    print(f"  Only P0:      {only_p0} ({100*only_p0/total:.1f}%)")
    print(f"  Only P1:      {only_p1} ({100*only_p1/total:.1f}%)")
    print(f"  Both empty:   {both_empty} ({100*both_empty/total:.1f}%)")
    
    print("\nClasses with most P1 usage (both present):")
    for label, count in class_both.most_common(10):
        print(f"  Class {label}: {count}")
    
    return {
        "both_present": both_present,
        "only_p0": only_p0,
        "only_p1": only_p1,
        "both_empty": both_empty,
    }

# Analyze both splits
print("=== TRAIN ===")
occ_trn = analyze_person_occupancy(x_trn, trn_labels, trn_names)

print("\n=== VAL ===")
occ_val = analyze_person_occupancy(x_val, val_labels, val_names)